In [1]:
!pip install --quiet opensearch-py
!pip install --quiet requests_auth_aws_sigv4

## Create Index

In [2]:
from opensearchpy import OpenSearch, RequestsHttpConnection
import json
from requests_auth_aws_sigv4 import AWSSigV4

In [3]:
MAX_ERRORS = 10
OPENSEARCH_DOMAIN_FQDN = 'vpc-int-use1-opensearch-ml-c32nkeiaudrckcr2ep7jghefje.us-east-1.es.amazonaws.com'
INDEX_NAME = 'search-data-titan-embed-3'
HEADERS = {
    'Content-Type': 'application/json',
    'Accept-Encoding': 'gzip',
}
auth = AWSSigV4('es')

client = OpenSearch(
    hosts=[{'host': OPENSEARCH_DOMAIN_FQDN, 'port':443}],
    http_auth=auth,
    use_ssl=True,
    connection_class=RequestsHttpConnection
)

In [4]:
# Create index
is_exist = client.indices.exists(index=INDEX_NAME)

if not is_exist:
    index_json_path = "opensearch_artifact/index.json"
    with open(index_json_path, "r") as file:
        index_body = json.load(file)
    
    response = client.indices.create(index=INDEX_NAME, body=index_body)
else:
    # Check index
    query={
        "query": {
            "match_all": {}
        }
    }
    print(client.search(index=INDEX_NAME, body=query))
    print("-"*50)
    print("Index already exists")

{'took': 97, 'timed_out': False, '_shards': {'total': 5, 'successful': 5, 'skipped': 0, 'failed': 0}, 'hits': {'total': {'value': 0, 'relation': 'eq'}, 'max_score': None, 'hits': []}}
--------------------------------------------------
Index already exists


### Add items

In [5]:
from src.bedrock import get_bearer_token
get_bearer_token()

Bearer token set as BEARER_TOKEN_STR global variable


In [6]:
from src.bedrock import get_titan_response, get_cohere_response

raw_data_json_path ="opensearch_artifact/raw-qna-data.json"
with open( raw_data_json_path, 'r') as f:
    records = json.load(f)

for record in records:
    text_info = "\n\n".join([record['description_s'], record['description_txt_edgeNgram'], record['metadata_s']])
    # Titan Embedding
    response_dict = get_titan_response(text_info)
    # # Cohere embedding
    # response_dict = get_cohere_response(text_info)
    
    embed_vec = response_dict["body"]["embedding"]
    record['vec_embedding'] = embed_vec

    response = client.index(index=INDEX_NAME, body=record)

/home/ec2-user/anaconda3/envs/python3/lib/python3.10/site-packages/urllib3/connectionpool.py:1097: InsecureRequestWarning: Unverified HTTPS request is being made to host 'aigateway-amrs-nonprod.oneadp.com'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(
/home/ec2-user/anaconda3/envs/python3/lib/python3.10/site-packages/urllib3/connectionpool.py:1097: InsecureRequestWarning: Unverified HTTPS request is being made to host 'aigateway-amrs-nonprod.oneadp.com'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(
/home/ec2-user/anaconda3/envs/python3/lib/python3.10/site-packages/urllib3/connectionpool.py:1097: InsecureRequestWarning: Unverified HTTPS request is being made to host 'aigateway-amrs-nonprod.oneadp.com'. Adding certificate verification is strongly advised. See: https://urllib3.readthe